# HW2：用 LoRA 教 LLM 寫支票

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/HW2-LoRA-Check-Writer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
!pip install -q -U transformers peft trl
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['zh_numbers.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

在 `05-LoRA-SFT` 中，我們教模型把中文數字轉成阿拉伯數字。這次反過來：

**把阿拉伯數字轉成支票上的大寫中文數字**，例如 `3052` → `參仟零伍拾貳`，`10050` → `壹萬零伍拾`。

這比上次難：模型必須正確處理「零」和「萬」的位置。

規則
* 基礎模型只能用 `Qwen/Qwen3-0.6B`（可以用 LoRA 或其他 PEFT 方法，也可以全參數微調）
* 訓練資料可以自己用 `zh_numbers.py` 產生，數量不限
* 測試時用 greedy decoding (`do_sample=False`)，答案必須**完全相同**才算對
* 必須在 Colab T4 上完成，總訓練時間不超過 30 分鐘
* 繳交：notebook（含執行結果）

可以嘗試的方向
* 資料量、數字分布（多一點含「零」的數字？）
* LoRA 的 `r`、`target_modules`、learning rate、epoch
* prompt 的設計
* 讓模型先「思考」(例如先把數字拆成萬、千、百、十位) 再回答？

In [ ]:
import random
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from zh_numbers import num2zh_upper, random_number
device = "cuda" if torch.cuda.is_available() else "cpu"
for n in [5, 10, 105, 3052, 10050, 10000005, 10100100]:
    print(n, num2zh_upper(n))

## 測試資料（請勿修改）

In [ ]:
test_rng = random.Random(2026)
test_numbers = [random_number(test_rng) for _ in range(300)]
test_answers = [num2zh_upper(n) for n in test_numbers]

## 你的程式碼

完成 `predict(numbers)`：輸入 int 的 list，回傳模型產生的字串 list。

In [ ]:
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")

PROMPT = "把下面的數字轉成支票用的大寫中文數字，只輸出答案：{}"

# TODO: 產生訓練資料，用 SFTTrainer + LoRA 訓練 (參考 05-LoRA-SFT)

@torch.no_grad()
def predict(numbers, batch_size=50):
    tokenizer.padding_side = "left"
    model.eval()
    results = []
    for i in range(0, len(numbers), batch_size):
        texts = [tokenizer.apply_chat_template([{"role": "user", "content": PROMPT.format(n)}], tokenize=False,
                                               add_generation_prompt=True, enable_thinking=False)
                 for n in numbers[i:i + batch_size]]
        inputs = tokenizer(texts, padding=True, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        results += [tokenizer.decode(o[inputs.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]
    return results

## 評分（請勿修改）

In [ ]:
preds = predict(test_numbers)
acc = sum(p == a for p, a in zip(preds, test_answers)) / len(test_answers)
for n, p, a in list(zip(test_numbers, preds, test_answers))[:10]:
    print(f"{n:>10} {p}  {'✅' if p == a else '❌ ' + a}")
print(f"test accuracy = {acc:.4f}")

## 評分方式（40 分）

* 98% 以上：40 分
* 95%：35 分
* 90%：28 分
* 80%：20 分
* 50%：10 分

（中間的分數線性內插）